In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
from pydub import AudioSegment
from IPython.display import Audio
import logging 
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from src.helper import *
from src.model import *
from src.model_sota import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
OUTPUT_PATH = "output/"
PREDS_PATH = "output/preds/"
MODELS_PATH = "models/"
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BEST_MODEL_PATH = "models/hybrid_audio_classifier_mixup__cross_entropy__adamw__lr_0_0001__epochs_20__bs_16.pth"
BEST_MODEL_CLASS = HybridAudioClassifier(num_classes=2, use_spec_augment=True).to(device)

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)
if not os.path.exists(PREDS_PATH):
    os.mkdir(PREDS_PATH)
if not os.path.exists(MODELS_PATH):
    os.mkdir(MODELS_PATH)

In [4]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_06_analyzing_model_result_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting {BEST_MODEL_PATH} result analysis...")

### Preparing the data loader

In [5]:
logger.info(f"preparing the dataloader...")

train_ds = AudioFolderDataset(os.path.join(DATA_PATH, "training/"))
test_ds = AudioFolderDataset(os.path.join(DATA_PATH, "testing/"))
holdout_ds = AudioFolderDataset(os.path.join(DATA_PATH, "holdout/"))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Inspect one batch
for X, y, path in train_loader:
    logger.info(f"batch input shape: {X.shape}")
    logger.info(f"batch labels shape: {y.shape}")
    break

### Loading the model

In [6]:
logger.info(f"loading the trained model...")

model = BEST_MODEL_CLASS
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

logger.info(f"trained model loaded")

### Evaluating test predictions

In [7]:
logger.info(f"generating test predictions...")

model.eval()
raw_outputs, y_true, y_pred, y_probs = [], [], [], []
with torch.no_grad():
    for X_batch, y_batch, path_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)  # raw logits

        # Apply softmax along class dimension to get probabilities
        probs = F.softmax(outputs, dim=1)

        # Store outputs and probabilities
        raw_outputs.append(outputs.cpu())
        y_probs.append(probs.cpu())

        # Get predicted class (argmax)
        preds = probs.argmax(dim=1).cpu().numpy()
        y_true.extend(y_batch.numpy())
        y_pred.extend(preds)

raw_outputs = torch.cat(raw_outputs).numpy()
y_probs = torch.cat(y_probs).numpy()

#get score summary
logger.info(f"model test f1-score: {f1_score(y_true, y_pred)}")
logger.info(f"model classification report: \n{classification_report(y_true, y_pred, digits=5)}")

In [16]:
df_summary = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred,
    'y_probs_1': [p[1] for p in y_probs],
})
df_summary['correct_prediction'] = np.where(
    df_summary['y_true']==df_summary['y_pred'],
    1,
    0
)
df_summary.head(3)

,y_true,y_pred,y_probs_1,correct_prediction
0,0,0,0.020915,1
1,0,0,0.017502,1
2,0,0,0.051578,1


In [17]:
df_summary.describe()

,y_true,y_pred,y_probs_1,correct_prediction
count,11710.000000,11710.000000,11710.000000,11710.000000
mean,0.404184,0.402306,0.406043,0.996926
std,0.490754,0.490384,0.474158,0.055363
min,0.000000,0.000000,0.000612,0.000000
25%,0.000000,0.000000,0.007679,1.000000
50%,0.000000,0.000000,0.027944,1.000000
75%,1.000000,1.000000,0.993082,1.000000
max,1.000000,1.000000,0.999968,1.000000


In [19]:
print("probability distribution of correct predictions")
df_summary.loc[df_summary['correct_prediction']==1, 'y_probs_1'].describe()

probability distribution of correct predictions


count    11674.000000
mean         0.406175
std          0.474710
min          0.000612
25%          0.007654
50%          0.027668
75%          0.993152
max          0.999968
Name: y_probs_1, dtype: float64

In [20]:
print("probability distribution of incorrect predictions")
df_summary.loc[df_summary['correct_prediction']==0, 'y_probs_1'].describe()

probability distribution of incorrect predictions


count    36.000000
mean      0.363002
std       0.233928
min       0.012270
25%       0.211010
50%       0.340528
75%       0.449164
max       0.989269
Name: y_probs_1, dtype: float64

Key takeaways: incorrect predictions tend to occur when the model is unsure about the prediction (probability between 20%-45%)

### Generating holdout predictions

In [26]:
logger.info(f"generating holdout predictions...")
model.eval()

model_name = BEST_MODEL_PATH.split('/')[-1].split('.')[0]
y_holdout_id = []
y_holdout_prob = []
y_holdout_pred = []
with torch.no_grad():
    for X_batch, y_batch, path_batch in holdout_loader:
        id = [p.split("/")[-1].split(".")[0]+".wav" for p in path_batch]
        y_holdout_id.extend(id)

        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = F.softmax(outputs, dim=1)
        y_holdout_prob.append(probs.cpu())
        preds = outputs.argmax(dim=1).cpu().numpy()
        y_holdout_pred.extend(preds)
y_holdout_prob = torch.cat(y_holdout_prob).numpy()

preds_save_path = os.path.join(PREDS_PATH, f'{model_name}_preds_with_prob.csv')
df_holdout_preds = pd.DataFrame()
df_holdout_preds['id'] = y_holdout_id
df_holdout_preds['label'] = y_holdout_pred
df_holdout_preds['prob_1'] = [p[1] for p in y_holdout_prob] 
df_holdout_preds.to_csv(preds_save_path, index=False)
logger.info(f"{model_name} holdout preds saved to {preds_save_path}")

[WARN] Failed to load data/deep-detect/dataset/holdout/audio_02268.wav: Error opening 'data/deep-detect/dataset/holdout/audio_02268.wav': Format not recognised.
[WARN] Failed to load data/deep-detect/dataset/holdout/audio_00183.wav: Error opening 'data/deep-detect/dataset/holdout/audio_00183.wav': Format not recognised.


### Experiment combining manual evaluation on unsure predictions

In [7]:
model_name = BEST_MODEL_PATH.split('/')[-1].split('.')[0]
df_holdout_preds = pd.read_csv(os.path.join(PREDS_PATH, f'{model_name}_preds_with_prob.csv'))
df_holdout_preds.head(3)

,id,label,prob_1
0,audio_03185.wav,1,0.999957
1,audio_12096.wav,1,0.996585
2,audio_07471.wav,0,0.001877


In [21]:
MIN_PROB_THRESHOLD = 0.20
MAX_PROB_THRESHOLD = 0.90

df_unsure_preds = df_holdout_preds.loc[
    (
        (df_holdout_preds['prob_1']>=MIN_PROB_THRESHOLD) &
        (df_holdout_preds['prob_1']<=MAX_PROB_THRESHOLD)
    ) |
    #add samples with unrecognizable format
    (df_holdout_preds['id'].isin(['audio_02268.wav', 'audio_00183.wav']))
].copy().reset_index(drop=True)
print(df_unsure_preds.shape)
df_unsure_preds.sort_values('id')

(28, 3)


,id,label,prob_1
19,audio_00183.wav,1,0.999835
22,audio_00515.wav,0,0.214230
17,audio_01468.wav,0,0.407441
23,audio_01859.wav,0,0.215411
1,audio_02010.wav,0,0.236934
18,audio_02268.wav,1,0.999835
12,audio_02717.wav,0,0.218352
5,audio_02886.wav,1,0.879143
6,audio_03369.wav,1,0.728760
2,audio_03429.wav,0,0.343799


In [54]:
#check the audio sound
sample_wav = os.path.join(DATA_PATH, "holdout/audio_14291.wav")
Audio(sample_wav)

In [ ]:
#ADD MANUAL EVALUATION
#note : a lot of clicking sounds of microphones, which can give flags for fake/real
#note : 'audio_02268.wav', 'audio_00183.wav' are fakes

# version = "v1"
# MANUAL_EVAL_DICT = {
#     'audio_00183.wav':0,
#     'audio_00515.wav':0,
#     'audio_01468.wav':1,
#     'audio_01859.wav':1,
#     'audio_02010.wav':0,
#     'audio_02268.wav':0,
#     'audio_02717.wav':0,
#     'audio_02886.wav':1,
#     'audio_03369.wav':0,
#     'audio_03429.wav':1,
#     'audio_03842.wav':0,
#     'audio_03975.wav':1,
#     'audio_04040.wav':0,
#     'audio_04048.wav':1,
#     'audio_05162.wav':1,
#     'audio_06237.wav':0,
#     'audio_07003.wav':0,
#     'audio_07418.wav':1,
#     'audio_07660.wav':0,
#     'audio_08026.wav':1,
#     'audio_08879.wav':0,
#     'audio_11309.wav':0,
#     'audio_11559.wav':0,
#     'audio_12894.wav':0,
#     'audio_13096.wav':0,
#     'audio_13789.wav':0,
#     'audio_14089.wav':1,
#     'audio_14291.wav':1,
# }

version = "v2"
MANUAL_EVAL_DICT = {
    'audio_00183.wav':1,
    'audio_00515.wav':0,
    'audio_01468.wav':1,
    'audio_01859.wav':1,
    'audio_02010.wav':0,
    'audio_02268.wav':1,
    'audio_02717.wav':0,
    'audio_02886.wav':1,
    'audio_03369.wav':0,
    'audio_03429.wav':1,
    'audio_03842.wav':0,
    'audio_03975.wav':1,
    'audio_04040.wav':0,
    'audio_04048.wav':1,
    'audio_05162.wav':1,
    'audio_06237.wav':0,
    'audio_07003.wav':0,
    'audio_07418.wav':1,
    'audio_07660.wav':0,
    'audio_08026.wav':1,
    'audio_08879.wav':0,
    'audio_11309.wav':0,
    'audio_11559.wav':0,
    'audio_12894.wav':0,
    'audio_13096.wav':0,
    'audio_13789.wav':0,
    'audio_14089.wav':1,
    'audio_14291.wav':1,
}

In [61]:
manual_eval_preds_save_path = os.path.join(PREDS_PATH, f'{model_name}_preds_manual_eval_{version}.csv')
df_manual_eval = df_holdout_preds[['id', 'label']].copy().reset_index(drop=True)
df_manual_eval.loc[df_manual_eval['id'].isin(MANUAL_EVAL_DICT.keys()), 'label'] = df_manual_eval['id'].map(MANUAL_EVAL_DICT)
df_manual_eval.to_csv(manual_eval_preds_save_path, index=False)